<a href="https://colab.research.google.com/github/Jiyaulhaq1432/Data-Analysis-Air-quality-Analyser-web-app/blob/main/Beijing_AQA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div align="center">

# 🌫️ Beijing Air Quality Analyser
## CMP7005 PRAC1
**Cardiff Metropolitan University · School of Technologies**

</div>

---

| Section | Description |
|---|---|
| **1 · Setup** | Install packages and write `app.py` to disk |
| **2 · Upload** | Upload your Beijing Air Quality CSV files |
| **3 · App** | Launch the interactive Streamlit dashboard |
| **4 · Analysis** | Full EDA + ML pipeline inline in the notebook |


In [2]:
!git clone 'https://github.com/Jiyaulhaq1432/Data-Analysis-Air-quality-Analyser-web-app.git'

fatal: destination path 'Data-Analysis-Air-quality-Analyser-web-app' already exists and is not an empty directory.


In [3]:
import os
REPO_URL = 'https://github.com/Jiyaulhaq1432/Data-Analysis-Air-quality-Analyser-web-app.git'
if not os.path.isdir('Data-Analysis-Air-quality-Analyser-web-app'):
    os.system(f'git clone {REPO_URL}')
    os.chdir('Data-Analysis-Air-quality-Analyser-web-app')
    os.system('git checkout main')
else:
    os.chdir('Data-Analysis-Air-quality-Analyser-web-app')
    os.system('git checkout main')
    os.system('git pull')
    os.chdir('/content')


print('\nDataset files available:')
for f in sorted(os.listdir('Data-Analysis-Air-quality-Analyser-web-app/Data Files')):
    if f.endswith('.csv'):
        size_mb = os.path.getsize(f'Data-Analysis-Air-quality-Analyser-web-app/Data Files/{f}') / 1_048_576
        print(f'   📄 {f}  ({size_mb:.1f} MB)')



Dataset files available:
   📄 PRSA_Data_Aotizhongxin_20130301-20170228.csv  (2.7 MB)
   📄 PRSA_Data_Changping_20130301-20170228.csv  (2.6 MB)
   📄 PRSA_Data_Dingling_20130301-20170228.csv  (2.6 MB)
   📄 PRSA_Data_Dongsi_20130301-20170228.csv  (2.5 MB)
   📄 PRSA_Data_Guanyuan_20130301-20170228.csv  (2.6 MB)
   📄 PRSA_Data_Gucheng_20130301-20170228.csv  (2.5 MB)
   📄 PRSA_Data_Huairou_20130301-20170228.csv  (2.5 MB)
   📄 PRSA_Data_Nongzhanguan_20130301-20170228.csv  (2.7 MB)
   📄 PRSA_Data_Shunyi_20130301-20170228.csv  (2.5 MB)
   📄 PRSA_Data_Tiantan_20130301-20170228.csv  (2.5 MB)
   📄 PRSA_Data_Wanliu_20130301-20170228.csv  (2.5 MB)
   📄 PRSA_Data_Wanshouxigong_20130301-20170228.csv  (2.7 MB)


---
## Section 1 — Setup

This section installs any packages not pre-installed in Colab and writes the
`app.py` Streamlit application to disk.

### 1.1 Install Required Packages

`streamlit` is not bundled with Colab; all other packages (`pandas`, `numpy`,
`scikit-learn`, `seaborn`, `plotly`) are already available.

In [4]:
import subprocess, sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "streamlit"],
    check=True,
)
print("✅ All packages ready")


✅ All packages ready


### 1.2 Write `app.py` to Disk

The code below writes the complete Streamlit application source directly to
`/content/app.py`.  No base64 encoding — you can read and edit the file freely.

**App pages:**
- 🏠 **Home** — project overview and dataset summary  
- 📋 **Dataset** — raw data preview, column info, missing-value chart  
- 📍 **EDA** — univariate, bivariate and multivariate analysis tabs  
- 📊 **Visualisations** — 10+ chart types (time series, heatmap, wind rose, …)  
- 🧪 **Modelling** — regression and classification with multiple models  
- 📝 **Report** — PM2.5 insights, correlations and references

In [5]:
# ── Write app.py ─────────────────────────────────────────────────────────────
import os, textwrap

APP_SOURCE = '''import streamlit as st
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import io
import glob
import os
import warnings
warnings.filterwarnings("ignore")

# ── ML imports ────────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, classification_report, confusion_matrix,
    roc_curve, auc, f1_score,
)
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                               RandomForestClassifier, GradientBoostingClassifier)
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.svm import SVR, SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

# ── Page config ───────────────────────────────────────────────────────────────
st.set_page_config(
    page_title="Beijing Air Quality Analyser",
    page_icon="🌫️",
    layout="wide",
    initial_sidebar_state="expanded",
)

# ── Global CSS ────────────────────────────────────────────────────────────────
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=DM+Sans:wght@300;400;600;700&display=swap');

html, body, [class*="css"] { font-family: 'DM Sans', sans-serif; }
.main { background: #0d1117; }
.block-container { padding: 1.5rem 2rem; }
h1, h2, h3 { font-family: 'Space Mono', monospace !important; }

.hero-banner {
    background: linear-gradient(135deg, #0f2027 0%, #203a43 50%, #2c5364 100%);
    border-radius: 16px; padding: 2.5rem 2rem; margin-bottom: 1.5rem;
    border: 1px solid rgba(99,202,255,0.2);
    box-shadow: 0 8px 32px rgba(0,0,0,0.4);
}
.hero-title {
    font-family: 'Space Mono', monospace; font-size: 2.2rem; font-weight: 700;
    color: #63caff; margin: 0 0 0.5rem 0; letter-spacing: -1px;
}
.hero-sub { color: #8ab4c9; font-size: 1rem; margin: 0; }

.metric-card {
    background: linear-gradient(135deg, #1a2332 0%, #1e2d40 100%);
    border-radius: 12px; padding: 1.2rem;
    border: 1px solid rgba(99,202,255,0.15);
    text-align: center; margin: 0.3rem 0;
}
.metric-val { font-size: 1.8rem; font-weight: 700; color: #63caff; font-family: 'Space Mono', monospace; }
.metric-lbl { font-size: 0.75rem; color: #8ab4c9; text-transform: uppercase; letter-spacing: 1px; }

.section-header {
    font-family: 'Space Mono', monospace; font-size: 1.1rem; color: #63caff;
    border-left: 3px solid #63caff; padding-left: 0.75rem;
    margin: 1.5rem 0 1rem 0;
}
.info-box {
    background: rgba(99,202,255,0.08); border: 1px solid rgba(99,202,255,0.2);
    border-radius: 10px; padding: 1rem 1.2rem; margin: 0.8rem 0;
    color: #cdd9e5; font-size: 0.9rem;
}
.stButton>button {
    background: linear-gradient(135deg, #1a6b8a, #0d4f6b); color: white;
    border: 1px solid rgba(99,202,255,0.3); border-radius: 8px;
    font-family: 'Space Mono', monospace; font-size: 0.85rem;
    padding: 0.5rem 1.2rem; width: 100%; transition: all 0.2s;
}
.stButton>button:hover { border-color: #63caff; box-shadow: 0 0 12px rgba(99,202,255,0.3); }

[data-testid="stSidebar"] {
    background: #0d1822; border-right: 1px solid rgba(99,202,255,0.15);
}
.score-badge {
    display: inline-block;
    background: linear-gradient(135deg, #0f3d52, #1a6b8a);
    color: #63caff; font-family: 'Space Mono', monospace; font-size: 0.8rem;
    padding: 0.3rem 0.7rem; border-radius: 20px;
    border: 1px solid rgba(99,202,255,0.3); margin: 0.2rem;
}
</style>
""", unsafe_allow_html=True)

# ── Plot theme helper ─────────────────────────────────────────────────────────
PLOT_THEME = dict(
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(13,24,34,0.8)",
    font=dict(family="DM Sans", color="#cdd9e5"),
    margin=dict(l=40, r=20, t=50, b=40),
)

def style_fig(fig, title=""):
    fig.update_layout(**PLOT_THEME, title=dict(
        text=title, font=dict(family="Space Mono", size=14, color="#63caff")
    ))
    fig.update_xaxes(gridcolor="rgba(99,202,255,0.08)", linecolor="rgba(99,202,255,0.2)")
    fig.update_yaxes(gridcolor="rgba(99,202,255,0.08)", linecolor="rgba(99,202,255,0.2)")
    return fig


# ── Cached helpers ────────────────────────────────────────────────────────────
@st.cache_data(show_spinner=False)
def _load_and_combine(file_contents: list) -> pd.DataFrame | None:
    """Load and merge multiple CSV/Excel files into one DataFrame."""
    dfs = []
    for name, content in file_contents:
        try:
            nl = name.lower()
            if nl.endswith(".csv"):
                df = pd.read_csv(io.BytesIO(content))
            elif nl.endswith((".xlsx", ".xls")):
                df = pd.read_excel(io.BytesIO(content))
            else:
                continue
            station = name.replace("PRSA_Data_", "").split("_20")[0]
            if "station" not in df.columns:
                df["station"] = station
            dfs.append(df)
        except Exception as e:
            st.warning(f"Could not read {name}: {e}")
    return pd.concat(dfs, ignore_index=True) if dfs else None


@st.cache_data(show_spinner=False)
def load_default_data() -> pd.DataFrame | None:
    """
    Load CSV files that are already present in the repo.
    Looks in a 'data/' folder next to app.py, then falls back to the
    current working directory.  Returns None if no files are found.
    """
    base = os.path.dirname(os.path.abspath(__file__))
    # Try  <repo>/data/*.csv  then  <repo>/*.csv
    patterns = [
        os.path.join(base, "data", "*.csv"),
        os.path.join(base, "*.csv"),
    ]
    files = []
    for pat in patterns:
        files = glob.glob(pat)
        if files:
            break

    if not files:
        return None

    dfs = []
    for path in sorted(files):
        try:
            part = pd.read_csv(path)
            name = os.path.basename(path)
            station = name.replace("PRSA_Data_", "").split("_20")[0].replace(".csv", "")
            if "station" not in part.columns:
                part["station"] = station
            dfs.append(part)
        except Exception as e:
            st.warning(f"Could not read {os.path.basename(path)}: {e}")

    return pd.concat(dfs, ignore_index=True) if dfs else None


@st.cache_data(show_spinner=False)
def preprocess(_df: pd.DataFrame) -> pd.DataFrame:
    """Full preprocessing pipeline — cached so navigation stays instant."""
    df = _df.copy()
    dt_cols = [c for c in ["year", "month", "day", "hour"] if c in df.columns]
    if len(dt_cols) == 4:
        df["datetime"] = pd.to_datetime(df[dt_cols])
        df["season"] = df["month"].map({
            12: "Winter", 1: "Winter", 2: "Winter",
            3: "Spring",  4: "Spring",  5: "Spring",
            6: "Summer",  7: "Summer",  8: "Summer",
            9: "Autumn",  10: "Autumn", 11: "Autumn",
        })
        df["day_of_week"] = df["datetime"].dt.day_name()
        df["is_weekend"] = (df["datetime"].dt.dayofweek >= 5).astype(int)
    if "PM2.5" in df.columns:
        bins   = [0, 12, 35.4, 55.4, 150.4, 250.4, float("inf")]
        labels = ["Good", "Moderate", "USG", "Unhealthy", "Very Unhealthy", "Hazardous"]
        df["AQI_Cat"] = pd.cut(df["PM2.5"], bins=bins, labels=labels)
    if "wd" in df.columns:
        df["wd"] = df["wd"].astype("category")
    return df


def get_numeric(df):     return df.select_dtypes(include=np.number).columns.tolist()
def get_categorical(df): return df.select_dtypes(include=["object", "category"]).columns.tolist()


# ═══════════════════════════════════════════════════════════════════════════════
# SIDEBAR
# ═══════════════════════════════════════════════════════════════════════════════
with st.sidebar:
    st.markdown("""
    <div style='text-align:center; padding:1rem 0 1.5rem'>
        <div style='font-family:Space Mono;font-size:1.5rem;color:#63caff;'>🌫️ AQA</div>
        <div style='color:#8ab4c9;font-size:0.75rem;letter-spacing:2px;'>AIR QUALITY ANALYSER</div>
    </div>
    """, unsafe_allow_html=True)

    page = st.radio(
        "NAVIGATE",
        ["🏠 Home", "📋 Dataset", "📍 EDA", "📊 Visualisations", "🧪 Modelling", "📝 Report"],
        label_visibility="visible",
    )

    st.markdown("---")
    st.markdown(
        "<div style='color:#8ab4c9;font-size:0.75rem;font-family:Space Mono;'>UPLOAD DATA</div>",
        unsafe_allow_html=True,
    )
    uploaded_files = st.file_uploader(
        "Upload CSV/Excel files",
        type=["csv", "xlsx", "xls"],
        accept_multiple_files=True,
        label_visibility="collapsed",
    )
    st.markdown(
        "<div class='info-box'>Optionally upload your own files to override the default dataset."
        " Supports CSV &amp; Excel.</div>",
        unsafe_allow_html=True,
    )

# ── Resolve active DataFrame ──────────────────────────────────────────────────
# Priority: uploaded files > repo default data
df_raw = None
df     = None

if uploaded_files:
    # User uploaded something — use that
    file_contents = [(f.name, f.read()) for f in uploaded_files]
    df_raw = _load_and_combine(file_contents)
else:
    # Fall back to the CSV files already in the repo
    df_raw = load_default_data()

if df_raw is not None:
    df = preprocess(df_raw)


# ═══════════════════════════════════════════════════════════════════════════════
# PAGE: HOME
# ═══════════════════════════════════════════════════════════════════════════════
if page == "🏠 Home":
    st.markdown("""
    <div class='hero-banner'>
        <div class='hero-title'>Beijing Air Quality<br>Analysis Platform</div>
        <p class='hero-sub'>CMP7005 · Programming for Data Analysis · Cardiff Metropolitan University</p>
    </div>
    """, unsafe_allow_html=True)

    c1, c2, c3, c4 = st.columns(4)
    for col, (val, lbl) in zip(
        [c1, c2, c3, c4],
        [("12", "Monitoring Stations"), ("35,064", "Hourly Records/Station"),
         ("11", "Variables"), ("4 Years", "2013 – 2017")],
    ):
        col.markdown(
            f"<div class='metric-card'><div class='metric-val'>{val}</div>"
            f"<div class='metric-lbl'>{lbl}</div></div>",
            unsafe_allow_html=True,
        )

    st.markdown("<div class='section-header'>📋 Project Overview</div>", unsafe_allow_html=True)
    st.markdown("""
    <div class='info-box'>
    This platform supports the full data analysis pipeline for the Beijing PM2.5 Air Quality
    dataset as required by <b>CMP7005 PRAC1</b>. Navigate through the five sections using the sidebar:<br><br>
    <b>📋 Dataset</b> – Inspect raw data, check data types and missing values.<br>
    <b>📍 EDA</b> – Automated Exploratory Data Analysis with statistical summaries.<br>
    <b>📊 Visualisations</b> – 10+ interactive chart types: distributions, correlations, temporal plots.<br>
    <b>🧪 Modelling</b> – Select regression or classification models, tune and evaluate performance.<br>
    <b>📝 Report</b> – Summary of findings and key insights.
    </div>
    """, unsafe_allow_html=True)

    st.markdown("<div class='section-header'>🌐 Dataset Variables</div>", unsafe_allow_html=True)
    col1, col2 = st.columns(2)
    with col1:
        st.markdown("""
**Air Quality Pollutants**
| Variable | Description |
|---|---|
| PM2.5 | Fine particulate matter (μg/m³) |
| PM10 | Coarse particulate matter (μg/m³) |
| SO2 | Sulphur dioxide (μg/m³) |
| NO2 | Nitrogen dioxide (μg/m³) |
| CO | Carbon monoxide (μg/m³) |
| O3 | Ozone (μg/m³) |
        """)
    with col2:
        st.markdown("""
**Meteorological Variables**
| Variable | Description |
|---|---|
| TEMP | Temperature (°C) |
| PRES | Atmospheric pressure (hPa) |
| DEWP | Dew point temperature (°C) |
| RAIN | Precipitation (mm) |
| WSPM | Wind speed (m/s) |
        """)

    if df is not None:
        source = "uploaded files" if uploaded_files else "default repo data"
        n_stations = df["station"].nunique() if "station" in df.columns else "?"
        st.success(
            f"✅ Data ready — **{len(df):,} rows** across **{n_stations} station(s)** "
            f"*(source: {source})*"
        )
    else:
        st.warning(
            "⚠️ No data found. Place CSV files in a `data/` folder next to app.py, "
            "or upload files using the sidebar."
        )


# ═══════════════════════════════════════════════════════════════════════════════
# PAGE: DATASET
# ═══════════════════════════════════════════════════════════════════════════════
elif page == "📋 Dataset":
    st.markdown("<h2 style='color:#63caff;font-family:Space Mono'>Dataset Explorer</h2>",
                unsafe_allow_html=True)

    if df is None:
        st.markdown(
            "<div class='info-box'>⚠️ No data found. Place CSV files in the <b>data/</b> folder "
            "next to app.py, or upload files using the sidebar.</div>",
            unsafe_allow_html=True,
        )
    else:
        num_cols = get_numeric(df)

        c1, c2, c3, c4 = st.columns(4)
        for col, (val, lbl) in zip(
            [c1, c2, c3, c4],
            [(f"{len(df):,}", "Rows"), (str(df.shape[1]), "Columns"),
             (f"{df.isnull().sum().sum():,}", "Missing Values"),
             (f"{df.duplicated().sum():,}", "Duplicates")],
        ):
            col.markdown(
                f"<div class='metric-card'><div class='metric-val'>{val}</div>"
                f"<div class='metric-lbl'>{lbl}</div></div>",
                unsafe_allow_html=True,
            )

        st.markdown("<div class='section-header'>Raw Data Preview</div>", unsafe_allow_html=True)
        n = st.slider("Rows to display", 5, 100, 20)
        st.dataframe(df.head(n), use_container_width=True)

        st.markdown("<div class='section-header'>Column Information</div>", unsafe_allow_html=True)
        info_df = pd.DataFrame({
            "Column":   df.columns,
            "Dtype":    df.dtypes.values,
            "Non-Null": df.count().values,
            "Null":     df.isnull().sum().values,
            "Null %":   (df.isnull().mean() * 100).round(2).values,
            "Unique":   df.nunique().values,
        })
        st.dataframe(info_df, use_container_width=True)

        st.markdown("<div class='section-header'>Statistical Summary</div>", unsafe_allow_html=True)
        st.dataframe(
            df[num_cols].describe().T.style.background_gradient(cmap="Blues"),
            use_container_width=True,
        )

        st.markdown("<div class='section-header'>Missing Values Chart</div>", unsafe_allow_html=True)
        miss = df[num_cols].isnull().mean().sort_values(ascending=False)
        miss = miss[miss > 0]
        if len(miss) > 0:
            fig = px.bar(miss, orientation="h",
                         labels={"index": "Column", "value": "Missing %"},
                         color=miss.values, color_continuous_scale="Blues")
            fig = style_fig(fig, "Missing Values (%)")
            st.plotly_chart(fig, use_container_width=True)
        else:
            st.success("✅ No missing values detected in numeric columns!")

        if "station" in df.columns:
            st.markdown("<div class='section-header'>Station Distribution</div>",
                        unsafe_allow_html=True)
            vc  = df["station"].value_counts()
            fig = px.bar(vc, labels={"index": "Station", "value": "Record Count"},
                         color=vc.values, color_continuous_scale="Blues")
            fig = style_fig(fig, "Records per Station")
            st.plotly_chart(fig, use_container_width=True)

        st.markdown("<div class='section-header'>Download Cleaned Dataset</div>",
                    unsafe_allow_html=True)
        buf = io.BytesIO()
        df.to_csv(buf, index=False)
        st.download_button("⬇️ Download as CSV", buf.getvalue(), "cleaned_data.csv", "text/csv")


'''

with open("/content/app.py", "w", encoding="utf-8") as f:
    f.write(APP_SOURCE)

size = os.path.getsize("/content/app.py")
print("✅  app.py written  (" + str(size) + " bytes  ·  " + str(size//1024) + " KB)")

✅  app.py written  (17935 bytes  ·  17 KB)
